# 5.1 · 逻辑回归 + 评估指标 / Logistic Regression + Metrics

> **课程定位 / Where this fits**
> **Part 5 第 1 课, 分类的"Hello World"**。0.8/0.11/2.9 都遇到过它的碎片(sigmoid 导数、交叉熵、伯努利 MLE); 4.7 GLM 说它是"伯努利+logit"特例。这一课**正式**讲透 + **顺带建立整套分类评估指标**(混淆矩阵/precision/recall/F1/ROC/PR)——因为没有指标就无法评估任何分类器。
> The Hello World of classification, plus the full metrics toolkit (confusion matrix / precision / recall / F1 / ROC / PR) since no classifier can be evaluated without them.

> 💡 **面试相关 / Interview-relevant**
> - "逻辑回归为什么用 sigmoid + 交叉熵" ★★★★★（MLE 推导）
> - "为什么不用 MSE 做分类" ★★★★★（非凸 + 梯度消失, 0.11 见过）
> - "precision/recall/F1/ROC-AUC 区别和何时用" ★★★★★
> - "ROC-AUC vs PR-AUC" ★★★★（不平衡）
> - "逻辑回归是线性还是非线性模型" ★★★★（决策边界线性）

---

## 学习目标 / Learning Objectives
1. 从**伯努利 MLE** 推出逻辑回归(sigmoid + 交叉熵)。
2. 理解决策边界为何**线性**, 系数的**对数几率**解读。
3. 从零梯度下降实现 + 对照 sklearn。
4. **建立完整分类指标体系**: 混淆矩阵 / P/R/F1 / ROC-AUC / PR-AUC。
5. 理解**阈值**把概率变成决策(接 3.11/2.10)。

## 目录 / TOC
1. [从伯努利 MLE 到逻辑回归 ⭐](#1)
2. [为什么不用 MSE ⭐](#2)
3. [🩺 数据 + 从零实现](#3)
4. [对照 sklearn + 决策边界 + 系数解读](#4)
5. [混淆矩阵 + P/R/F1 ⭐](#5)
6. [ROC 曲线与 AUC ⭐](#6)
7. [PR 曲线: 不平衡的首选 ⭐](#7)
8. [阈值: 概率→决策](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 从伯努利 MLE 到逻辑回归 ⭐ / From Bernoulli MLE

**目标**: 预测二分类概率 $\Pr(y=1|\mathbf{x})$。线性回归 $\mathbf{x}^\top\mathbf{w}$ 值域无限, 不能当概率(4.7 GLM 讲过)。用 **sigmoid** 把它压到 $[0,1]$：

$$p = \sigma(\mathbf{x}^\top\mathbf{w}) = \frac{1}{1 + e^{-\mathbf{x}^\top\mathbf{w}}}$$

**损失从哪来**: 伯努利 MLE(2.9)。单样本似然 $p^{y}(1-p)^{1-y}$, 负对数似然 = **二元交叉熵**：

$$J(\mathbf{w}) = -\frac{1}{n}\sum_i \big[y_i \log p_i + (1-y_i)\log(1-p_i)\big]$$

**梯度异常简洁**(0.8 节算过, sigmoid 导数的妙处)：
$$\nabla_{\mathbf{w}} J = \frac{1}{n}\mathbf{X}^\top(\boldsymbol{p} - \mathbf{y})$$

和线性回归梯度 $\mathbf{X}^\top(\hat{\mathbf{y}}-\mathbf{y})$ **形式完全一样**——这不是巧合, 是 GLM(4.7) 指数族的共同结构。
The gradient has the same form as linear regression's — not a coincidence, but the shared GLM structure.


<a id="2"></a>
## 2. 为什么不用 MSE ⭐ / Why Not MSE

**面试高频**(0.11/2.9 都铺垫过)。用 MSE + sigmoid 做分类的两个致命问题：

1. **非凸**: $\frac{1}{2}(\sigma(\mathbf{x}^\top\mathbf{w}) - y)^2$ 关于 $\mathbf{w}$ **非凸** → 梯度下降陷局部最优。交叉熵则是**凸**的 → 全局最优。
2. **梯度消失**(0.11 节实测过): MSE+sigmoid 的梯度含 $\sigma'(z) = \sigma(1-\sigma)$, 在模型**自信地错**时($z\to\pm\infty$) $\sigma'\to 0$ → 梯度消失 → 学不动。交叉熵梯度是干净的 $(p - y)$, **永不消失**。

→ **分类永远用交叉熵**, 这是 0.11/2.9 的结论在此处的落地。
Classification always uses cross-entropy: convex + non-vanishing gradient.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

data = load_breast_cancer()
X, y = data.data, data.target          # 0=恶性, 1=良性
print(f"Breast Cancer: {X.shape}, 类别分布 {np.bincount(y)} (良性 {y.mean():.0%})")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)        # 正则化的逻辑回归需缩放 (3.4)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


<a id="3"></a>
## 3. 从零实现 / From Scratch


In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def fit_logreg(X, y, lr=0.1, n_iter=1000):
    Xb = np.c_[np.ones(len(X)), X]      # 加偏置
    w = np.zeros(Xb.shape[1])
    for _ in range(n_iter):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(y)   # 交叉熵梯度 = Xᵀ(p-y)/n
        w -= lr * grad
    return w

w = fit_logreg(Xtr, y_tr)
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)   # 默认 L2; 数据近可分, 无正则系数会爆

# 系数量级取决于正则强度, 直接比"差异"无意义; 比方向(相关性)和预测一致性才公平
corr = np.corrcoef(w[1:], sk.coef_[0])[0, 1]
p_test = sigmoid(np.c_[np.ones(len(Xte)), Xte] @ w)
agree = ((p_test > 0.5).astype(int) == sk.predict(Xte)).mean()
print(f"从零 vs sklearn 系数方向相关性: {corr:.3f}  (≈1 表示学到同一方向)")
print(f"两者测试集预测一致率:        {agree:.3f}")
print(f"从零模型 test 准确率:        {((p_test > 0.5).astype(int) == y_te).mean():.3f}")
print("注: 系数绝对量级由正则强度决定, 无正则会让近可分数据系数发散; 故比方向不比大小")


<a id="4"></a>
## 4. 对照 sklearn + 决策边界 + 系数解读 / sklearn & Coefficients

**决策边界是线性的**: $\Pr(y=1)=0.5 \iff \mathbf{x}^\top\mathbf{w}=0$, 这是一个超平面。所以逻辑回归是**线性分类器**——尽管 sigmoid 是非线性的, **边界仍是直线/超平面**。


In [ ]:
# 2D 可视化决策边界 (取两个特征) / 2D decision boundary
from sklearn.linear_model import LogisticRegression
X2 = Xtr[:, [0, 3]]      # mean radius, mean area (标准化后)
clf2 = LogisticRegression().fit(X2, y_tr)

xx, yy = np.meshgrid(np.linspace(X2[:,0].min(), X2[:,0].max(), 200),
                     np.linspace(X2[:,1].min(), X2[:,1].max(), 200))
Z = clf2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.5)
ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)   # 决策边界=直线
ax.scatter(X2[:,0], X2[:,1], c=y_tr, cmap="RdBu", edgecolor="k", s=20)
plt.colorbar(cs, label="P(良性)"); ax.set_xlabel("mean radius"); ax.set_ylabel("mean area")
ax.set_title("逻辑回归决策边界 = 直线 (P=0.5 等高线)\nsigmoid 非线性, 但边界线性 → 线性分类器")
plt.tight_layout(); plt.show()


In [ ]:
# 系数 = 对数几率 (log-odds) 解读 / coefficients as log-odds
sk = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
coef = pd.Series(sk.coef_[0], index=data.feature_names).sort_values(key=abs, ascending=False)
print("系数 Top 5 (标准化后, log-odds 视角):")
for name, c in coef.head(5).items():
    print(f"  {name:<22} {c:+.2f}  → 该特征+1σ, odds×{np.exp(c):.2f}")
print("\n💡 系数解读: 逻辑回归系数是'对数几率'(log-odds)")
print("   exp(系数) = 该特征+1单位时几率(odds)的倍数变化 — 这是逻辑回归独特的可解释性")


<a id="5"></a>
## 5. 混淆矩阵 + P/R/F1 ⭐ / Confusion Matrix & P/R/F1

**没有指标就无法评估分类器**——这套指标是 Part 5 全程的语言。从**混淆矩阵**开始：

| | 预测正 | 预测负 |
|---|---|---|
| **真实正** | TP (真阳) | FN (假阴, 漏报) |
| **真实负** | FP (假阳, 误报) | TN (真阴) |

由它派生（3.11 见过, 这里系统化）：
$$\text{Precision} = \frac{TP}{TP+FP} \;(\text{报警准不准}), \quad \text{Recall} = \frac{TP}{TP+FN} \;(\text{抓全不全})$$
$$F_1 = 2\cdot\frac{P\cdot R}{P+R} \;(\text{P/R 调和均值})$$

**precision vs recall 的权衡 = 业务选择**: 癌症筛查要高 recall(别漏诊), 垃圾邮件要高 precision(别误杀正常邮件)。


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, accuracy_score)
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)
y_pred = clf.predict(Xte)

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["恶性","良性"], yticklabels=["恶性","良性"], ax=ax)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title("混淆矩阵")
plt.tight_layout(); plt.show()

print(f"accuracy  = {accuracy_score(y_te, y_pred):.3f}")
print(f"precision = {precision_score(y_te, y_pred):.3f}")
print(f"recall    = {recall_score(y_te, y_pred):.3f}")
print(f"F1        = {f1_score(y_te, y_pred):.3f}")
print(f"\n💡 癌症场景: 这里 1=良性, 漏判恶性(预测良性实际恶性)是 FN, 最危险")
print("   实际应把'恶性'设为正类并优化 recall — 别漏诊")


<a id="6"></a>
## 6. ROC 曲线与 AUC ⭐ / ROC Curve & AUC

**precision/recall 依赖一个固定阈值(0.5)**。ROC 曲线**扫所有阈值**, 画 **TPR(=recall) vs FPR(=FP/(FP+TN))**：

- 每个阈值对应曲线上一点; 阈值从 1→0, 曲线从左下→右上
- **完美分类器**: 直奔左上角(TPR=1, FPR=0)
- **随机猜**: 对角线
- **AUC (曲线下面积)**: 模型把**随机一对(正,负)样本排对**的概率。0.5=瞎猜, 1=完美

**AUC 的妙处**: 衡量**排序能力**, 与阈值无关、与类别比例无关。
ROC-AUC measures ranking ability — threshold-free and (somewhat) class-ratio-independent.


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

y_proba = clf.predict_proba(Xte)[:, 1]
fpr, tpr, thresh = roc_curve(y_te, y_proba)
auc = roc_auc_score(y_te, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f"逻辑回归 (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"k--", label="随机猜 (AUC=0.5)")
ax.scatter([0],[1], c="g", s=100, marker="*", label="完美分类器", zorder=5)
ax.set_xlabel("FPR (假阳率)"); ax.set_ylabel("TPR (真阳率=recall)")
ax.legend(); ax.set_title(f"ROC 曲线 — AUC={auc:.3f}")
plt.tight_layout(); plt.show()
print(f"AUC = {auc:.3f} = 随机取一对(良性,恶性), 模型给良性更高分的概率")
print("AUC 衡量排序, 与阈值无关 → 比单一准确率信息丰富")


<a id="7"></a>
## 7. PR 曲线: 不平衡的首选 ⭐ / PR Curve

**ROC 的盲区**: 极端**不平衡**时, ROC 会**过度乐观**——因为 FPR 的分母 TN 巨大, 大量假阳也只让 FPR 微增, ROC 看着还很好。

**PR 曲线**(precision vs recall)在不平衡下更诚实——它**不看 TN**, 只关注正类。**不平衡分类(3.11/5.14)首选 PR-AUC**(= average precision)。


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# 人为制造极端不平衡看 ROC vs PR 的差异 / extreme imbalance
rng2 = np.random.default_rng(0)
n_pos = 30; n_neg = 2000
score_pos = rng2.normal(1.0, 1, n_pos)       # 正类分数略高
score_neg = rng2.normal(0.0, 1, n_neg)
y_imb = np.r_[np.ones(n_pos), np.zeros(n_neg)]
s_imb = np.r_[score_pos, score_neg]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fpr, tpr, _ = roc_curve(y_imb, s_imb)
axes[0].plot(fpr, tpr); axes[0].plot([0,1],[0,1],"k--")
axes[0].set_title(f"ROC (AUC={roc_auc_score(y_imb, s_imb):.2f}) — 看着不错")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")

prec, rec, _ = precision_recall_curve(y_imb, s_imb)
ap = average_precision_score(y_imb, s_imb)
axes[1].plot(rec, prec)
axes[1].axhline(n_pos/(n_pos+n_neg), color="k", ls="--", label=f"基线(正类比例={n_pos/(n_pos+n_neg):.1%})")
axes[1].set_title(f"PR (AP={ap:.2f}) — 暴露真实困难")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"1.5% 正类: ROC-AUC={roc_auc_score(y_imb, s_imb):.2f}(乐观) 但 PR-AUC={ap:.2f}(揭示真实困难)")
print("→ 不平衡看 PR-AUC; ROC 在极端不平衡下被海量 TN 掩盖了 precision 问题")


### 指标选择速查
| 场景 | 看什么指标 |
|---|---|
| 类别均衡 + 关心整体对错 | accuracy + F1 |
| 漏报代价高(癌症/欺诈漏抓) | **recall** |
| 误报代价高(垃圾邮件误杀) | **precision** |
| 综合排序能力(均衡) | **ROC-AUC** |
| 综合排序能力(不平衡) | **PR-AUC** ⭐ |
| 概率值要准(风控阈值) | 校准(5.15) |


<a id="8"></a>
## 8. 阈值: 概率→决策 / Threshold

逻辑回归输出**概率**, 默认 0.5 阈值变成 0/1 决策。但 0.5 是任意的(3.11 讲过)——**按业务成本调阈值**(2.10 期望损失思想)。


In [ ]:
# 调阈值看 P/R 权衡 / threshold tuning
thresholds = np.linspace(0.05, 0.95, 19)
precs = [precision_score(y_te, y_proba >= t, zero_division=1) for t in thresholds]
recs = [recall_score(y_te, y_proba >= t) for t in thresholds]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precs, "o-", label="precision")
ax.plot(thresholds, recs, "s-", label="recall")
ax.axvline(0.5, color="gray", ls="--", label="默认 0.5")
ax.set_xlabel("阈值"); ax.legend()
ax.set_title("阈值权衡: 阈值↑→precision↑recall↓; 按业务成本选")
plt.tight_layout(); plt.show()
print("阈值低→抓全(高recall)但误报多; 阈值高→报准(高precision)但漏报多")
print("癌症筛查应降阈值(宁可误报多查也别漏诊) — 阈值是部署旋钮, 由代价决定")


<a id="9"></a>
## 9. 小结 / Summary

```
逻辑回归: sigmoid(xᵀw) → 概率; 损失=交叉熵(=伯努利 MLE, 2.9); 梯度=Xᵀ(p-y)/n
不用 MSE: 非凸 + 梯度消失 → 交叉熵凸 + 梯度不消失(0.11)
决策边界线性 → 线性分类器(尽管 sigmoid 非线性); 系数=对数几率, exp=odds 倍数
评估指标体系 ⭐:
  混淆矩阵 → precision(报准) / recall(抓全) / F1(调和)
  ROC-AUC: 排序能力, 阈值无关; 均衡数据用
  PR-AUC: 不平衡首选(不看 TN)
  阈值: 概率→决策, 按业务成本调(2.10)
```

### 💡 面试速查
1. **逻辑回归 = sigmoid + 交叉熵**, 由伯努利 MLE 推出
2. **不用 MSE**: 非凸 + 梯度消失; 交叉熵凸 + 梯度 (p-y) 不消失
3. **决策边界线性** → 线性分类器; 系数 = log-odds
4. **precision vs recall**: 误报代价 vs 漏报代价的权衡
5. **ROC-AUC 均衡用, PR-AUC 不平衡用**(ROC 被海量 TN 蒙蔽)

### 下一节
**5.2 Softmax 回归**——逻辑回归处理二分类。多分类(3类以上)怎么办? Softmax 把 sigmoid 推广到 K 类, 输出概率分布。
